# 7-1. 훈련 파이프라인 — 손실 함수로 구분하기

LLM이 만들어지는 과정은 보통 일곱 단계로 나눠 부릅니다. 이름이 많아 헷갈리지만,
**무엇을 줄이는가(손실 함수)** 와 **어떤 토큰에 손실을 거는가(마스크)** 두 가지만 보면
경계가 선명해집니다.

| 단계 | 손실 형태 | 손실이 걸리는 토큰 | 준비할 것 |
|---|---|---|---|
| 사전훈련 pre-training | 다음 토큰 예측 | 모든 토큰 | 웹 대규모 코퍼스 |
| 중간훈련 mid-training | 다음 토큰 예측 | 모든 토큰 | 고품질 큐레이션 믹스 |
| CPT continued pre-training | 다음 토큰 예측 | 모든 토큰 | 도메인 코퍼스 + 리플레이 |
| **SFT 지시 파인튜닝** | 다음 토큰 예측 | **assistant 응답 + 종료 토큰** | 지시–응답 쌍 |
| DPO 선호 정렬 | 쌍별 로그오즈 | 해당 없음 | 선호쌍 (chosen/rejected) |
| RLVR 강화학습 | 보상 기대값 | 해당 없음 | 프롬프트 + 검증기 |
| 증류 distillation | KL 발산 | 학생이 만든 모든 토큰 | 프롬프트 + 교사 모델 |

앞의 **넷은 손실 형태가 같습니다.** 그중 앞의 셋은 마스크까지 같아서,
실제로 다른 것은 데이터와 학습률 위치뿐입니다.
목적함수가 **정말로 바뀌는 것은 DPO부터**입니다.

> 이 노트북은 그 표를 말로 외우는 대신, **네 가지 손실을 전부 직접 계산해서**
> 어디가 같고 어디가 다른지 숫자로 확인합니다.

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.abspath("../.."))   # SelfStudy/
sys.path.insert(0, os.path.abspath(".."))      # SelfStudy/Part4/

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from common import get_device, set_seed
from chat import IGNORE_INDEX

set_seed(1234)
device = get_device()
V = 32   # 장난감 어휘 크기

## 1. 다음 토큰 예측 — 앞의 네 단계가 공유하는 손실

로짓 `(B, T, V)` 와 라벨 `(B, T)` 를 한 칸 밀어서 교차 엔트로피를 겁니다.
사전훈련·중간훈련·CPT·SFT가 **전부 이 함수 하나**를 씁니다.

In [ ]:
def next_token_loss(logits, labels):
    """표준 LM 손실. labels 의 IGNORE_INDEX 자리는 자동으로 빠집니다."""
    shift_logits = logits[:, :-1, :].contiguous()   # 마지막 위치는 맞힐 다음 토큰이 없음
    shift_labels = labels[:, 1:].contiguous()       # 라벨을 한 칸 당김
    return F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=IGNORE_INDEX,     # <- 마스킹이 실제로 일어나는 지점
    )

B, T = 2, 12
logits = torch.randn(B, T, V)
labels = torch.randint(0, V, (B, T))

print("사전훈련/중간훈련/CPT  (모든 토큰):", f"{next_token_loss(logits, labels).item():.4f}")

# SFT: 앞 7토큰(system+user)을 가린다
sft_labels = labels.clone()
sft_labels[:, :7] = IGNORE_INDEX
print("SFT                    (응답만  ):", f"{next_token_loss(logits, sft_labels).item():.4f}")

n_all = (labels != IGNORE_INDEX)[:, 1:].sum().item()
n_sft = (sft_labels != IGNORE_INDEX)[:, 1:].sum().item()
print(f"\n손실 계산에 참여한 토큰 수: 전체 {n_all} -> SFT {n_sft}")
print("함수는 똑같습니다. 바뀐 것은 labels 에 찍힌 IGNORE_INDEX 뿐입니다.")

### 가린 토큰도 모델은 **읽습니다**

이게 가장 많이 헷갈리는 지점입니다. 마스킹은 `labels` 에만 하고
`input_ids` 는 건드리지 않습니다. 답은 앞의 시스템 메시지와 질문을 보고 쓰는 것이라서,
가린 부분이 입력에서 빠지면 무엇에 대한 답인지 알 수 없습니다.

> 가린다 = 「이 부분을 못 맞혔다고 혼내지 않는다」
> 가린다 ≠ 「보여주지 않는다」

아래에서 직접 확인합니다 — 앞부분을 가려도 **그래디언트는 거기까지 흐릅니다.**

In [ ]:
# 위치끼리 섞이지 않는 모델이면 이 실험이 성립하지 않습니다.
# 앞 토큰이 뒤 토큰의 예측에 쓰이는 통로 = 어텐션. 그래서 인과 어텐션 한 층을 둡니다.
emb  = torch.nn.Embedding(V, 16)
attn = torch.nn.MultiheadAttention(16, num_heads=2, batch_first=True)
head = torch.nn.Linear(16, V)

ids = torch.randint(0, V, (1, 10))
labs = ids.clone()
labs[:, :6] = IGNORE_INDEX          # 앞 6토큰은 손실에서 제외

h = emb(ids)
causal = torch.triu(torch.ones(10, 10, dtype=torch.bool), diagonal=1)   # 뒤를 못 보게
h, _ = attn(h, h, h, attn_mask=causal, need_weights=False)
loss = next_token_loss(head(h), labs)
loss.backward()

grad_per_pos = emb.weight.grad[ids[0]].abs().sum(-1)
print("위치별 입력 임베딩 그래디언트 크기:")
for i, (g, l) in enumerate(zip(grad_per_pos, labs[0])):
    tag = "가림(손실 제외)" if l.item() == IGNORE_INDEX else "학습"
    print(f"  pos {i:2d}  {tag:<14} |grad| = {g.item():.5f}")
print("\n-> 손실에서 뺀 자리에도 그래디언트가 0이 아닙니다.")
print("   뒤쪽 응답 토큰을 맞히는 데 앞의 질문을 '읽어서' 썼기 때문입니다.")
print("   마지막 위치만 0인 것은, 그 뒤에 맞힐 토큰이 없어서입니다.")

## 2. DPO — 여기서 목적함수가 바뀝니다

선호쌍 `(chosen, rejected)` 의 **로그확률 차이**를 키웁니다.
정답 토큰을 맞히는 문제가 아니라, 두 응답의 **우열**을 맞히는 문제로 바뀌었습니다.

$$\mathcal{L}_{\text{DPO}} = -\log\sigma\Big(\beta\big[(\log \pi_\theta(y_w) - \log \pi_{\text{ref}}(y_w)) - (\log \pi_\theta(y_l) - \log \pi_{\text{ref}}(y_l))\big]\Big)$$

`ref` 는 SFT가 끝난 시점의 모델을 얼려 둔 것입니다. 원형인 RLHF는 여기서
보상 모델을 따로 학습시켰고, DPO는 그 보상 모델을 없앤 간소화판입니다.

In [ ]:
def dpo_loss(pi_chosen, pi_rejected, ref_chosen, ref_rejected, beta=0.1):
    """네 개의 로그확률 합(시퀀스 단위)만 있으면 계산됩니다."""
    logits = beta * ((pi_chosen - ref_chosen) - (pi_rejected - ref_rejected))
    return -F.logsigmoid(logits).mean(), logits

ref_c, ref_r = torch.tensor([-20.0]), torch.tensor([-20.0])

for name, (pc, pr) in {
    "아직 구분 못 함      ": (-20.0, -20.0),
    "선호 쪽을 더 좋아함  ": (-18.0, -22.0),
    "거꾸로 배웠음        ": (-22.0, -18.0),
}.items():
    l, m = dpo_loss(torch.tensor([pc]), torch.tensor([pr]), ref_c, ref_r)
    print(f"{name} loss = {l.item():.4f}   margin = {m.item():+.3f}")

print("\n선호 쪽 확률이 올라갈수록 손실이 내려갑니다.")
print("'정답 토큰'이라는 개념 자체가 없습니다 — 그래서 표의 '손실이 걸리는 토큰'이 '해당 없음'입니다.")

## 3. RLVR — 보상 기대값

사람도 보상 모델도 아니고 **검증 코드**가 점수를 줍니다.
조건은 「추론 과제인가」가 아니라 **「코드로 채점되는가」** 입니다.

그래서 수학·코드만이 아니라 JSON 스키마 준수, 길이 제한, 금칙어,
툴 호출 성공 여부까지 전부 여기 들어갑니다.
반대로 페르소나·문체처럼 채점기를 짤 수 없는 것은 **선호 정렬 쪽 일**입니다.
(챕터 8에서 다룰 페르소나가 DPO 계열로 가는 이유입니다.)

In [ ]:
import re, json

def verifier_exact(pred, gold):      return float(pred.strip() == gold.strip())
def verifier_json(pred, _):
    try: json.loads(pred); return 1.0
    except Exception: return 0.0
def verifier_no_comma(pred, _):      return float("," not in pred)

samples = [
    ("42",              "42",   verifier_exact,    "정답 대조"),
    ('{"a": 1}',        None,   verifier_json,     "JSON 스키마"),
    ("쉼표, 넣음",       None,   verifier_no_comma, "형식 제약"),
    ("쉼표 없음",        None,   verifier_no_comma, "형식 제약"),
]
rewards = [v(p, g) for p, g, v, _ in samples]
for (p, _, _, kind), r in zip(samples, rewards):
    print(f"  [{kind:<10}] {p!r:<14} -> reward {r}")
print(f"\n배치 보상 기대값 E[r] = {sum(rewards)/len(rewards):.3f}   (이 값을 올리는 것이 목적)")

## 4. 증류 — KL 발산

정답을 사람이 아니라 **다른 모델(교사)** 이 줍니다.
하드 증류는 교사의 *답 문자열* 을 정답으로 쓰고(= 결국 SFT),
소프트 증류는 교사의 **확률분포 전체** 를 가져옵니다. (챕터7 퀴즈 8번)

온폴리시 증류는 **학생이 직접 쓴 문장** 을 교사가 토큰마다 채점하는 방식입니다.
강화학습이 에피소드마다 점수 하나를 받는 반면, 증류는 **토큰마다 점수**를 받습니다 —
같은 연산으로 훨씬 촘촘한 신호가 나오고, 2026년 프런티어 레시피(MOPD)가
강화학습 자리를 증류로 대체하기 시작한 이유입니다.

In [ ]:
teacher = torch.randn(1, 6, V)
student = teacher + 0.5 * torch.randn(1, 6, V)       # 교사와 조금 다른 학생

def soft_distill_loss(s_logits, t_logits, T=1.0):
    t_p = F.softmax(t_logits / T, dim=-1)
    s_logp = F.log_softmax(s_logits / T, dim=-1)
    return F.kl_div(s_logp, t_p, reduction="batchmean") * (T ** 2)

def reverse_kl(s_logits, t_logits):
    """온폴리시 증류가 쓰는 역KL — 학생 분포 기준으로 잰다."""
    s_p = F.softmax(s_logits, dim=-1)
    return (s_p * (F.log_softmax(s_logits, -1) - F.log_softmax(t_logits, -1))).sum(-1).mean()

print(f"소프트 증류 (정방향 KL) : {soft_distill_loss(student, teacher).item():.4f}")
print(f"온폴리시 증류 (역 KL)   : {reverse_kl(student, teacher).item():.4f}")
print(f"학생 = 교사 일 때       : {soft_distill_loss(teacher, teacher).item():.6f}  <- 0 으로 수렴")

# 하드 증류는 교사의 argmax 를 정답으로 쓰는 것 = 결국 SFT 와 같은 손실
hard = teacher.argmax(-1)
print(f"\n하드 증류 (= 다음 토큰 예측): {next_token_loss(student, hard).item():.4f}")

## 5. 학습률 곡선이 단계 이름의 절반이다

단계 이름의 상당수는 **학습률 곡선 위의 위치**를 가리키는 말입니다.

- **중간훈련(어닐링)** = 같은 런의 마지막 감쇠 구간. 학습률이 낮아 파라미터 변화 폭이 작고,
  그래서 여기 넣은 비싼 데이터가 가장 크게 반영됩니다.
- **CPT** = 감쇠가 끝나 공개된 베이스 위에서 **다시 올렸다가 다시 내리는** 것(re-warm → re-decay).
  두 번째 봉우리가 보이면 CPT입니다. (챕터7 퀴즈 3번)
- **SFT** = 낮은 값으로 **고정**. 크게 잡으면 형식은 빨리 익히는 대신
  사전훈련에서 배운 것을 잊습니다(catastrophic forgetting).

In [ ]:
plt.rcParams["font.family"] = "AppleGothic"      # 맥 한글 폰트 (윈도우: "Malgun Gothic")
plt.rcParams["axes.unicode_minus"] = False

def lr_curve(total, warm, stable, peak=3e-4, floor=3e-5):
    xs = []
    for t in range(total):
        if t < warm:                      xs.append(peak * t / warm)
        elif t < warm + stable:           xs.append(peak)
        else:
            p = (t - warm - stable) / max(1, total - warm - stable)
            xs.append(floor + 0.5 * (peak - floor) * (1 + math.cos(math.pi * p)))
    return xs

pre = lr_curve(1000, 50, 600)                        # 사전훈련 + 중간훈련(끝 감쇠)
cpt = [3e-5 + (1.2e-4 - 3e-5) * t / 40 for t in range(40)] + \
      [3e-5 + 0.5 * (1.2e-4 - 3e-5) * (1 + math.cos(math.pi * t / 260)) for t in range(260)]
sft = [2e-5] * 150

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(range(1000), pre, label="사전훈련 → 중간훈련(감쇠)", lw=2)
ax.plot(range(1000, 1300), cpt, label="CPT (재가열 → 재감쇠)", lw=2)
ax.plot(range(1300, 1450), sft, label="SFT (낮은 고정 LR)", lw=2.5)
ax.axvspan(650, 1000, alpha=.10, color="tab:blue")
ax.text(825, 2.6e-4, "중간훈련\n고품질 데이터", ha="center", fontsize=9)
ax.set_xlabel("step"); ax.set_ylabel("learning rate")
ax.legend(fontsize=9); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

## 6. 정리 — 내가 무엇을 준비해야 하는가

| 하려는 것 | 준비할 것 |
|---|---|
| SFT | 지시 **와** 응답 둘 다 |
| DPO | 응답 여러 개 **와 그 우열** |
| RLVR | 프롬프트 **와 검증기** |
| 증류 | 프롬프트 **와 교사 모델** (정답 응답 불필요) |

오른쪽으로 갈수록 사람이 쓰는 글이 줄고 모델이 만드는 몫이 늘어납니다.

일곱 칸을 전부, 이 순서대로 밟아야 하는 것은 아닙니다.
개인이 손댈 수 있는 현실적 범위는 **SFT와 DPO**, 그리고 PEFT를 얹은 그 변형입니다.

### 확인 문제
1. SFT와 CPT는 손실 함수가 같습니다. 그러면 둘을 가르는 것은 무엇입니까?
2. 페르소나 말투를 RLVR로 학습시키기 어려운 이유는?
3. 교사가 반드시 학생보다 큰 모델이어야 합니까?

> **다음** → `0702_InstructionTuning.ipynb`
> 표에서 SFT 줄의 「assistant 응답 + 종료 토큰만」이 실제로 어떻게 만들어지는지 봅니다.